In [1]:
# Cell 1 — Imports + path setup (no GPU)
import sys, os
sys.path.insert(0, "/workspace/shared/audit_validator")

import json
import numpy as np
import torch

print(f"PyTorch   : {torch.__version__}")
print(f"CUDA/ROCm : {torch.cuda.is_available()}")
print(f"GPU       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print("✅ Imports OK")

PyTorch   : 2.8.0+gitb2fb688
CUDA/ROCm : True
GPU       : 
✅ Imports OK


In [6]:
import subprocess

pkgs = ["faiss-cpu", "sentence-transformers", "pdfplumber", 
        "pypdf", "langchain", "langchain-community",
        "langchain-huggingface", "transformers", 
        "accelerate", "fpdf2", "tqdm"]

for pkg in pkgs:
    r = subprocess.run(["pip", "install", pkg, "-q", "--no-cache-dir"],
                       capture_output=True, text=True)
    print(f"{'✅' if r.returncode == 0 else '❌'} {pkg}")

✅ faiss-cpu
✅ sentence-transformers
✅ pdfplumber
✅ pypdf
✅ langchain
✅ langchain-community
✅ langchain-huggingface
✅ transformers
✅ accelerate
✅ fpdf2
✅ tqdm


In [7]:
# Cell 2 — Check constants loaded correctly (no GPU)
from src.constants import *

print(f"LLM Model  : {LLM_MODEL_ID}")
print(f"Embed Model: {EMBED_MODEL_ID}")
print(f"Dtype      : {TORCH_DTYPE}")
print(f"Temp       : {TEMPERATURE}")
print(f"Max tokens : {MAX_NEW_TOKENS}")
print("✅ Constants OK")

LLM Model  : Qwen/Qwen2.5-7B-Instruct
Embed Model: BAAI/bge-small-en-v1.5
Dtype      : torch.float16
Temp       : 0.05
Max tokens : 512
✅ Constants OK


In [8]:
# Cell 3 — Load + verify compliance rules
from src.rule_loader import load_all_rules, load_rules_as_text

rules      = load_all_rules()
rule_texts = load_rules_as_text(rules)

print(f"\nTotal rules     : {len(rules)}")
print(f"Rule text items : {len(rule_texts)}")
print(f"\n--- Sample Rule Text ---")
print(rule_texts[0]["text"])
print(f"\nFrameworks: {set(r['framework'] for r in rules)}")
print(f"Severities: {set(r['severity'] for r in rules)}")

Loaded 13 rules from 3 frameworks

Total rules     : 13
Rule text items : 13

--- Sample Rule Text ---
Rule ID: GDPR-001 | Framework: GDPR | Category: Data Collection | Severity: HIGH
Title: Lawful basis for processing
Description: Personal data must be processed on a lawful basis such as consent, contract, legal obligation, vital interests, public task, or legitimate interests.
Required clause: The document must state the legal basis under which personal data is collected and processed.
Keywords: consent, lawful basis, processing, personal data

Frameworks: {'GDPR', 'Insurance_Compliance', 'SOX'}
Severities: {'HIGH', 'MEDIUM', 'CRITICAL'}


In [9]:
# Cell 4 — Build FAISS index
from src.rag_pipeline import build_faiss_index, save_index, index_exists
from src.utils import log_gpu_memory, Timer

log_gpu_memory("Before building index")

with Timer("Full index build"):
    index, rule_texts_indexed = build_faiss_index(rule_texts)

save_index(index, rule_texts_indexed)
log_gpu_memory("After building index")

print(f"\nIndex stats:")
print(f"  Total vectors : {index.ntotal}")
print(f"  Is trained    : {index.is_trained}")
print("✅ FAISS index built and saved!")

[GPU Memory] Before building index
  Allocated : 0.00 GB
  Reserved  : 0.00 GB
  Total     : 206.14 GB
  Free      : 206.14 GB
Loading embedding model on: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: BAAI/bge-small-en-v1.5


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Embedding rules] 23.774s
FAISS index built: 13 vectors | dim=384
[Full index build] 23.774s
Index saved to: /workspace/shared/audit_validator/data/vector_store
[GPU Memory] After building index
  Allocated : 0.21 GB
  Reserved  : 0.25 GB
  Total     : 206.14 GB
  Free      : 205.89 GB

Index stats:
  Total vectors : 13
  Is trained    : True
✅ FAISS index built and saved!


In [10]:
# Cell 5 — Test retrieval quality
from src.rag_pipeline import retrieve_top_rules
from src.utils import Timer

test_queries = [
    "We collect personal data and retain it for 3 years after which it is deleted",
    "Management conducts internal control assessment over financial reporting quarterly",
    "Customer identity verification is performed before account opening using KYC process",
    "All audit workpapers are retained for a period of seven years"
]

print("=== Retrieval Quality Test ===\n")
for query in test_queries:
    print(f"Query: {query[:80]}...")
    with Timer("Retrieval"):
        results = retrieve_top_rules(query, index, rule_texts_indexed, top_k=3)
    print("Top matches:")
    for r in results:
        print(f"  [{r['similarity_score']:.3f}] {r['rule_id']} | {r['framework']} | {r['rule']['title']}")
    print()

=== Retrieval Quality Test ===

Query: We collect personal data and retain it for 3 years after which it is deleted...
[Retrieval] 0.029s
Top matches:
  [0.704] GDPR-002 | GDPR | Retention period
  [0.661] SOX-003 | SOX | Financial record retention
  [0.654] GDPR-003 | GDPR | Right to access and erasure

Query: Management conducts internal control assessment over financial reporting quarter...
[Retrieval] 0.015s
Top matches:
  [0.747] SOX-001 | SOX | Internal control over financial reporting
  [0.692] SOX-002 | SOX | Independent auditor attestation
  [0.598] GDPR-004 | GDPR | Breach notification

Query: Customer identity verification is performed before account opening using KYC pro...
[Retrieval] 0.011s
Top matches:
  [0.728] INS-003 | Insurance_Compliance | AML/KYC compliance
  [0.587] GDPR-004 | GDPR | Breach notification
  [0.587] GDPR-003 | GDPR | Right to access and erasure

Query: All audit workpapers are retained for a period of seven years...
[Retrieval] 0.009s
Top matches:
  

In [11]:
# Cell 6 — Load Qwen2.5-7B
from src.validator import get_llm_pipeline
from src.utils import log_gpu_memory, Timer

log_gpu_memory("Before LLM load")

with Timer("Model load"):
    pipe = get_llm_pipeline()

log_gpu_memory("After LLM load")
print("✅ Qwen2.5-7B ready!")

[GPU Memory] Before LLM load
  Allocated : 0.21 GB
  Reserved  : 0.25 GB
  Total     : 206.14 GB
  Free      : 205.89 GB
Loading Qwen/Qwen2.5-7B-Instruct ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device set to use cuda:0


[GPU Memory] After LLM load
  Allocated : 15.45 GB
  Reserved  : 15.58 GB
  Total     : 206.14 GB
  Free      : 190.56 GB
LLM ready: Qwen/Qwen2.5-7B-Instruct
[Model load] 25.523s
[GPU Memory] After LLM load
  Allocated : 15.45 GB
  Reserved  : 15.58 GB
  Total     : 206.14 GB
  Free      : 190.56 GB
✅ Qwen2.5-7B ready!


In [12]:
# Cell 7 — Test single clause validation 
from src.validator import validate_clause
from src.rag_pipeline import retrieve_top_rules
from src.utils import Timer
import json

test_clause = """
1. DATA RETENTION POLICY
The Company retains all customer personal data indefinitely for business analytics 
purposes. Data is stored in encrypted cloud servers and shared with our marketing 
partners for targeted advertising. Customers may request data deletion by writing 
to our headquarters.
"""

# Retrieve relevant rules
rules_for_clause = retrieve_top_rules(test_clause, index, rule_texts_indexed, top_k=3)

print("Rules retrieved for this clause:")
for r in rules_for_clause:
    print(f"  {r['rule_id']} | {r['rule']['title']} | score={r['similarity_score']}")

# Validate
print("\nRunning LLM validation...")
with Timer("Full validation"):
    result = validate_clause(test_clause, rules_for_clause)

print("\n=== VALIDATION RESULT ===")
print(json.dumps(result, indent=2))

Rules retrieved for this clause:
  GDPR-002 | Retention period | score=0.7714
  GDPR-003 | Right to access and erasure | score=0.6654
  SOX-003 | Financial record retention | score=0.6639

Running LLM validation...
[LLM inference] 20.46s
[Full validation] 20.46s

=== VALIDATION RESULT ===
{
  "validations": [
    {
      "rule_id": "GDPR-002",
      "framework": "GDPR",
      "severity": "HIGH",
      "status": "NON_COMPLIANT",
      "confidence_score": 85,
      "evidence": "\"The Company retains all customer personal data indefinitely\"",
      "gap": "\"A clear data retention period or deletion schedule must be defined\"",
      "recommendation": "Define a specific data retention period based on the original purpose of data collection."
    },
    {
      "rule_id": "GDPR-003",
      "framework": "GDPR",
      "severity": "HIGH",
      "status": "COMPLIANT",
      "confidence_score": 95,
      "evidence": "\"Customers may request data deletion by writing to our headquarters\"",
    

In [13]:
# Cell 8 — Parse and display result clearly (no GPU)
def display_validation_result(result: dict, clause_preview: str = ""):
    if clause_preview:
        print(f"Document Clause: {clause_preview[:100]}...\n")
    
    validations = result.get("validations", [])
    metrics     = result.get("metrics", {})
    
    print(f"{'='*60}")
    print(f"Rules checked : {len(validations)}")
    print(f"Tokens used   : {metrics.get('total_tokens', 'N/A')}")
    print(f"{'='*60}\n")
    
    status_icons = {
        "COMPLIANT"      : "✅",
        "NON_COMPLIANT"  : "❌",
        "PARTIAL"        : "⚠️",
        "NOT_APPLICABLE" : "➖"
    }
    
    for v in validations:
        icon = status_icons.get(v.get("status", ""), "❓")
        print(f"{icon} {v.get('rule_id')} | {v.get('framework')} | Severity: {v.get('severity')}")
        print(f"   Status     : {v.get('status')}")
        print(f"   Confidence : {v.get('confidence_score')}/100")
        print(f"   Evidence   : {str(v.get('evidence',''))[:120]}")
        if v.get("gap"):
            print(f"   Gap        : {v.get('gap')}")
        if v.get("recommendation"):
            print(f"   Fix        : {v.get('recommendation')}")
        print()

display_validation_result(result, test_clause)

Document Clause: 
1. DATA RETENTION POLICY
The Company retains all customer personal data indefinitely for business a...

Rules checked : 3
Tokens used   : 447

❌ GDPR-002 | GDPR | Severity: HIGH
   Status     : NON_COMPLIANT
   Confidence : 85/100
   Evidence   : "The Company retains all customer personal data indefinitely"
   Gap        : "A clear data retention period or deletion schedule must be defined"
   Fix        : Define a specific data retention period based on the original purpose of data collection.

✅ GDPR-003 | GDPR | Severity: HIGH
   Status     : COMPLIANT
   Confidence : 95/100
   Evidence   : "Customers may request data deletion by writing to our headquarters"

➖ SOX-003 | SOX | Severity: HIGH
   Status     : NOT_APPLICABLE
   Confidence : 100/100
   Evidence   : No mention of financial records or audit workpapers in the provided document.



In [14]:
# Cell 9 — End to end pipeline test with full document
from src.document_parser import parse_document, chunk_document
from src.validator import validate_full_document
from src.utils import Timer
import json, time

# Use the sample doc from Day 1
doc_path = "/workspace/shared/audit_validator/data/sample_docs/sample_contract.txt"
doc      = parse_document(doc_path)
chunks   = chunk_document(doc, chunk_size=150, overlap=30)

print(f"Document : {doc['filename']}")
print(f"Words    : {doc['word_count']}")
print(f"Chunks   : {len(chunks)}")

# Run full validation
t_start = time.time()
full_result = validate_full_document(
    chunks[:4],   # limit to 4 chunks to save GPU time in testing
    index,
    rule_texts_indexed,
    top_k=3
)
total_time = round(time.time() - t_start, 2)

print(f"\n=== FULL DOCUMENT RESULT ===")
print(f"Chunks validated   : {full_result['total_chunks']}")
print(f"Rules checked      : {full_result['total_rules_checked']}")
print(f"Total tokens used  : {full_result['total_tokens']}")
print(f"Avg latency/chunk  : {full_result['avg_latency_sec']}s")
print(f"Total time         : {total_time}s")

# Count by status
from collections import Counter
statuses = Counter(v["status"] for v in full_result["validations"])
print(f"\nCompliance summary:")
for status, count in statuses.items():
    print(f"  {status}: {count}")

Document : sample_contract.txt
Words    : 109
Chunks   : 1

Validating chunk 1/1...
[LLM inference] 6.575s

=== FULL DOCUMENT RESULT ===
Chunks validated   : 1
Rules checked      : 3
Total tokens used  : 592
Avg latency/chunk  : 6.57s
Total time         : 6.59s

Compliance summary:
  COMPLIANT: 2
  PARTIAL: 1


In [15]:
# Cell 10 — Save results + log metrics
import json
from datetime import datetime

# Save full result
output_path = f"/workspace/shared/audit_validator/outputs/audit_reports/test_result_{datetime.now().strftime('%Y%m%d_%H%M')}.json"
with open(output_path, "w") as f:
    json.dump(full_result, f, indent=2)
print(f"Result saved: {output_path}")

# Log Day 2 metrics for slide 4
day2_metrics = {
    "day"              : 2,
    "date"             : datetime.now().isoformat(),
    "model_llm"        : LLM_MODEL_ID,
    "model_embed"      : EMBED_MODEL_ID,
    "total_rules"      : len(rules),
    "faiss_index_size" : index.ntotal,
    "sample_tokens"    : full_result["total_tokens"],
    "avg_latency_sec"  : full_result["avg_latency_sec"],
    "chunks_tested"    : full_result["total_chunks"],
    "rules_checked"    : full_result["total_rules_checked"]
}
with open("/workspace/shared/audit_validator/logs/day2_metrics.json", "w") as f:
    json.dump(day2_metrics, f, indent=2)

print(f"\n📊 Day 2 Metrics (save these for Slide 4):")
print(json.dumps(day2_metrics, indent=2))
print("\n✅ Day 2 complete! RAG pipeline fully working.")

Result saved: /workspace/shared/audit_validator/outputs/audit_reports/test_result_20260609_1539.json

📊 Day 2 Metrics (save these for Slide 4):
{
  "day": 2,
  "date": "2026-06-09T15:39:36.986396",
  "model_llm": "Qwen/Qwen2.5-7B-Instruct",
  "model_embed": "BAAI/bge-small-en-v1.5",
  "total_rules": 13,
  "faiss_index_size": 13,
  "sample_tokens": 592,
  "avg_latency_sec": 6.57,
  "chunks_tested": 1,
  "rules_checked": 3
}

✅ Day 2 complete! RAG pipeline fully working.
